# Stream B — 02 Stage 1 Representation Model

This is the updated Stage 1 notebook for the cleaned project.

It trains a small GPU MLP over Stream B CLIP-derived features on NewsCLIPpings train/val.

Note: because local NewsCLIPpings images are unavailable, this stage currently uses text embeddings + deterministic image-key fallback features unless you later integrate the original NewsCLIPpings pre-extracted image embeddings.


In [1]:
from pathlib import Path
import sys, os, json, pickle, time, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root. Run this notebook inside eve-rag.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    MANIFESTS, PROCESSED_IMAGES, STREAM_B_CACHE,
    TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST,
    VERITE_TEST_MANIFEST, MMFAKEBENCH_VAL_MANIFEST, MMFAKEBENCH_TEST_MANIFEST,
    ALL_MANIFEST, FIGURES, TABLES
)

STREAM_B_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("STREAM_B_CACHE:", STREAM_B_CACHE)


ROOT: E:\eve-rag
STREAM_B_CACHE: E:\eve-rag\cache\stream_b


In [2]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

# Tuned for 8 GB RTX 4060 Ti. Reduce if OOM.
IMAGE_BATCH_SIZE = 48
TEXT_BATCH_SIZE = 256
TRAIN_BATCH_SIZE = 512
NUM_WORKERS = 0  # Windows-friendly. Try 2 later if stable.
print("IMAGE_BATCH_SIZE:", IMAGE_BATCH_SIZE)
print("TEXT_BATCH_SIZE:", TEXT_BATCH_SIZE)
print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0
IMAGE_BATCH_SIZE: 48
TEXT_BATCH_SIZE: 256
TRAIN_BATCH_SIZE: 512


In [3]:
FEATURE_DIR = STREAM_B_CACHE / "features"
STAGE1_DIR = STREAM_B_CACHE / "lora_stage1"
STAGE1_DIR.mkdir(parents=True, exist_ok=True)

features_path = FEATURE_DIR / "clip_features_all.csv"
if not features_path.exists():
    raise FileNotFoundError(f"Missing {features_path}. Run clip_features_updated.ipynb first.")

df = pd.read_csv(features_path, low_memory=False)
print(df.shape)
df.groupby(["dataset", "split"]).size()


(97361, 13)


dataset        split
mmfakebench    test     10000
               val       1000
newsclippings  test      7264
               train    71072
               val       7024
verite         test      1001
dtype: int64

In [4]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

FEATURE_COLS = [
    "clip_cosine", "l2_dist", "absdiff_mean", "absdiff_max",
    "prod_mean", "text_norm", "image_norm", "has_real_image_emb"
]

train_df = df[(df["dataset"] == "newsclippings") & (df["split"] == "train")].copy()
val_df = df[(df["dataset"] == "newsclippings") & (df["split"] == "val")].copy()

print("train:", train_df.shape, "val:", val_df.shape)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS].fillna(0).values).astype(np.float32)
y_train = train_df["label"].astype(int).values.astype(np.float32)
X_val = scaler.transform(val_df[FEATURE_COLS].fillna(0).values).astype(np.float32)
y_val = val_df["label"].astype(int).values.astype(np.float32)

pickle.dump({"scaler": scaler, "feature_cols": FEATURE_COLS}, open(STAGE1_DIR / "stage1_preprocess.pkl", "wb"), protocol=4)


train: (71072, 13) val: (7024, 13)


In [5]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(ArrayDataset(X_train, y_train), batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))
val_loader = DataLoader(ArrayDataset(X_val, y_val), batch_size=TRAIN_BATCH_SIZE*2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))

class StreamBStage1(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = StreamBStage1(len(FEATURE_COLS)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
scaler_amp = torch.amp.GradScaler("cuda", enabled=USE_AMP)

EPOCHS = 15
PATIENCE = 4
best_auc = -1
bad_epochs = 0
history = []


In [6]:
def predict_loader(loader):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(xb)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(ys)

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt)
        scaler_amp.update()
        losses.append(float(loss.detach().cpu()))

    val_prob, val_y = predict_loader(val_loader)
    val_pred = (val_prob >= 0.5).astype(int)
    auc = roc_auc_score(val_y, val_prob)
    acc = accuracy_score(val_y, val_pred)
    f1 = f1_score(val_y, val_pred)
    rec = {"epoch": epoch, "train_loss": float(np.mean(losses)), "val_auc": auc, "val_acc": acc, "val_f1": f1}
    history.append(rec)
    print(rec)

    if auc > best_auc:
        best_auc = auc
        bad_epochs = 0
        torch.save(model.state_dict(), STAGE1_DIR / "stage1_model.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

pd.DataFrame(history).to_csv(STAGE1_DIR / "stage1_history.csv", index=False)
print("Best AUC:", best_auc)


{'epoch': 1, 'train_loss': 0.6221670148183973, 'val_auc': 0.7292436751184355, 'val_acc': 0.6509111617312073, 'val_f1': 0.6239263803680981}
{'epoch': 2, 'train_loss': 0.6023185656225081, 'val_auc': 0.7301179149521848, 'val_acc': 0.651623006833713, 'val_f1': 0.6263551687280501}
{'epoch': 3, 'train_loss': 0.6010954534407142, 'val_auc': 0.7311830071061274, 'val_acc': 0.6543280182232346, 'val_f1': 0.6289731051344744}
{'epoch': 4, 'train_loss': 0.6009669123793677, 'val_auc': 0.7310236121777077, 'val_acc': 0.6563211845102506, 'val_f1': 0.6366646598434678}
{'epoch': 5, 'train_loss': 0.6009787702731949, 'val_auc': 0.7312616911234375, 'val_acc': 0.6550398633257403, 'val_f1': 0.6320425208807897}
{'epoch': 6, 'train_loss': 0.6011109656567196, 'val_auc': 0.731333240474572, 'val_acc': 0.654755125284738, 'val_f1': 0.6326314194818967}
{'epoch': 7, 'train_loss': 0.6004602004298203, 'val_auc': 0.7313961957959951, 'val_acc': 0.6544703872437357, 'val_f1': 0.6312110621486097}
{'epoch': 8, 'train_loss': 0.6

In [7]:
# Save Stage 1 predictions for all rows.
model.load_state_dict(torch.load(STAGE1_DIR / "stage1_model.pt", map_location=DEVICE))
model.eval()

prep = pickle.load(open(STAGE1_DIR / "stage1_preprocess.pkl", "rb"))
X_all = prep["scaler"].transform(df[prep["feature_cols"]].fillna(0).values).astype(np.float32)

all_loader = DataLoader(ArrayDataset(X_all, np.zeros(len(X_all), dtype=np.float32)), batch_size=TRAIN_BATCH_SIZE*2, shuffle=False, num_workers=NUM_WORKERS)

probs = []
with torch.no_grad():
    for xb, _ in all_loader:
        xb = xb.to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(xb)
        probs.append(torch.sigmoid(logits).cpu().numpy())

out = df[["pair_id", "dataset", "split", "label"]].copy()
out["stage1_score"] = np.concatenate(probs)
out.to_csv(STAGE1_DIR / "stage1_scores_all.csv", index=False)
out.head()


,pair_id,dataset,split,label,stage1_score
0,nc_620701_0,newsclippings,train,0,0.332764
1,nc_620701_1,newsclippings,train,1,0.813965
2,nc_1569820_0,newsclippings,train,0,0.641602
3,nc_1569820_1,newsclippings,train,1,0.572266
4,nc_1490080_0,newsclippings,train,0,0.486084


In [8]:
print("=" * 60)
print("Stage 1 complete.")
print("Next: lora_stage2_fixed_updated.ipynb")
print("=" * 60)


Stage 1 complete.
Next: lora_stage2_fixed_updated.ipynb
